In [ ]:
from ultralytics import YOLO
import torch
import pandas as pd
import ast
import numpy as np
torch.cuda.empty_cache()
torch.cuda.synchronize()

model = YOLO('yolov8n.pt')
yaml_path = r"yolo_dataset\dataset.yaml"
model.train(
    data=yaml_path,
    epochs=5,
    imgsz=640,
    batch=16,
    name='gesture_detector',
     project=".",
    pretrained = True,
    degrees = 10.0,
    freeze= 10
)

In [ ]:
row = pd.read_csv(r"yolo_dataset\df.csv")

row['bbox'] = row['bbox'].apply(ast.literal_eval)
row['label'] = row['label'].apply(ast.literal_eval)

random = np.random.randint(0, len(row))

In [ ]:
from ultralytics import YOLO
from PIL import Image
import numpy as np
import matplotlib.patches as patches
import matplotlib.pyplot as plt

model = YOLO(r'runs/detect/gesture_detector-7/weights/best.pt')

random = np.random.randint(0, len(row))
path = row.iloc[random]['path']
img = Image.open(path)

results = model(img, save=False, verbose=False)

for box in results[0].boxes:
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    conf = box.conf[0].item()
    class_id = int(box.cls[0].item())
    label = model.names[class_id]
    print(f"Klasa: {label}, pewność: {conf:.2f}, bbox: ({x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f})")


fig, ax = plt.subplots()
img = plt.imread(path)

ax.imshow(img)
h_img, w_img = img.shape[:2]

for idx,box in enumerate(results[0].boxes):
    x, y, w, h = box.xywhn[0].tolist()
    class_id = int(box.cls[0].item())
    label = model.names[class_id]
    x *= w_img
    w *= w_img
    y *= h_img
    h *= h_img
    rect = patches.Rectangle(
        (x - w/2, y - h/2), w, h,
        facecolor='none', linewidth=2, edgecolor='red'
    )
    ax.add_patch(rect)
    ax.text(x,y-5,label, color='red')

plt.axis('off')
plt.show()